In [4]:
# orb_15min_retest_from_parquet_with_equity_losses.py
# ORB 15m Retest strategy (same logic), plus equity curve with losses/drawdown + optional bear shading.

import warnings
warnings.filterwarnings("ignore")

import os, glob, time
from datetime import timedelta
from typing import List, Tuple
import numpy as np
import pandas as pd

# =============================
# USER PARAMETERS
# =============================
PARQUET_DIR = r"C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet"

ASSUME_NAIVE_TIMESTAMPS_ARE_UTC = True
TZ = "America/New_York"
REG_SESSION_START = "09:30"
REG_SESSION_END   = "16:00"

# ORB logic (kept identical to your Yahoo version)
RETEST_CONFIRM_CLOSE = True
MAX_RETEST_MIN = 120

# Risk / exits (kept identical)
R_MULTIPLES = [1.0, 2.0]
POSITION_SIZE_DOLLARS = 10_000
SLIPPAGE_BPS = 1.0
FEES_PER_TRADE = 0.00

# Batch / persistence
AUTOSAVE_EVERY = 25
SLEEP_BETWEEN_TICKERS = 0.2

# Output files
TRADES_CSV_ALL = "orb_trades_sp500.csv"
EQUITY_CSV_ALL = "orb_equity_sp500.csv"

# Plotting
PLOT_SAVE_DIR = "orb_plots"
PLOT_SHOW = False                 # set True to display interactively
PLOT_EXAMPLES_PER_TICKER = 0      # set >0 to also save day-level ORB charts
SHADE_BEAR_WITH_SPY = True        # requires SPY.parquet in the folder

# =============================
# Helpers
# =============================
STD_MAP = {"open":"Open","high":"High","low":"Low","close":"Close","volume":"Volume"}

def _standardize_ohlcv_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    lower = {c.lower(): c for c in df.columns}
    for k, std in STD_MAP.items():
        if k in lower:                      rename[lower[k]] = std
        elif k.capitalize() in df.columns:  rename[k.capitalize()] = std
        elif k.upper() in df.columns:       rename[k.upper()] = std
    out = df.rename(columns=rename)
    drop_cols = [c for c in out.columns if str(c).lower().startswith("adj")]
    return out.drop(columns=drop_cols, errors="ignore")

def _choose_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.index, pd.DatetimeIndex):
        return df
    for cand in ["EventAt","Datetime","datetime","Timestamp","timestamp","Date","date","Time","time"]:
        if cand in df.columns:
            df[cand] = pd.to_datetime(df[cand], errors="coerce")
            df = df.set_index(cand)
            break
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("No datetime index/column found.")
    return df

def _infer_median_interval_minutes(idx: pd.DatetimeIndex) -> float:
    if len(idx) < 2: return np.nan
    d = (idx[1:] - idx[:-1]).total_seconds() / 60.0
    return float(np.median(d)) if len(d) else np.nan

def _resample_to_15m_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    m = _infer_median_interval_minutes(df.index)
    if np.isnan(m): return pd.DataFrame()
    if m < 15 - 1e-6:
        agg = {"Open":"first","High":"max","Low":"min","Close":"last"}
        if "Volume" in df.columns: agg["Volume"] = "sum"
        out = (df.sort_index()
                 .resample("15T", label="right", closed="right")
                 .agg(agg)
                 .dropna(subset=["Open","High","Low","Close"]))
        return out
    elif abs(m - 15) <= 1e-6:
        return df.sort_index()
    else:
        return pd.DataFrame()

def load_parquet_15m(path: str, tz: str = TZ) -> pd.DataFrame:
    df = pd.read_parquet(path, engine="pyarrow")
    df = _choose_dt_index(df).sort_index()
    if df.index.tz is None:
        if ASSUME_NAIVE_TIMESTAMPS_ARE_UTC:
            df = df.tz_localize("UTC").tz_convert(tz)
        else:
            df = df.tz_localize(tz)
    else:
        df = df.tz_convert(tz)
    df = _standardize_ohlcv_columns(df)
    for need in ["Open","High","Low","Close"]:
        if need not in df.columns:
            raise ValueError(f"{os.path.basename(path)} missing required column: {need}")
    df = _resample_to_15m_if_needed(df)
    if df.empty: return df
    df = df.between_time(REG_SESSION_START, REG_SESSION_END)
    return df.sort_index()

def sessionize(df_15: pd.DataFrame) -> pd.Series:
    return pd.to_datetime(df_15.index.date)

# =============================
# ORB logic (same as your Yahoo version)
# =============================
def compute_opening_range(df_15: pd.DataFrame) -> pd.DataFrame:
    df = df_15.copy()
    for col in ["ORH","ORL"]:
        if col not in df.columns:
            df[col] = np.nan
    df["Session"] = sessionize(df)
    first_bar_idx = df.groupby("Session").head(1).index
    df.loc[first_bar_idx, "ORH"] = df.loc[first_bar_idx, "High"].astype(float)
    df.loc[first_bar_idx, "ORL"] = df.loc[first_bar_idx, "Low"].astype(float)
    df[["ORH","ORL"]] = df.groupby("Session")[["ORH","ORL"]].ffill()
    return df

def _apply_slippage(price: float, bps: float, side: str) -> float:
    factor = 1 + (bps/10000.0)
    return price * factor if side == "buy" else price / factor

def backtest_orb_retest(
    df_15: pd.DataFrame,
    max_retest_min: int = MAX_RETEST_MIN,
    r_targets: List[float] = R_MULTIPLES,
    dollars: float = POSITION_SIZE_DOLLARS,
    slippage_bps: float = SLIPPAGE_BPS,
    fees: float = FEES_PER_TRADE,
    retest_confirm_close: bool = RETEST_CONFIRM_CLOSE
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df_15.copy()
    df["Session"] = sessionize(df)
    sessions = df["Session"].unique()

    trades = []
    for ses in sessions:
        sdf = df[df["Session"] == ses].copy()
        if len(sdf) < 3:
            continue

        orh = sdf["ORH"].iloc[0]
        orl = sdf["ORL"].iloc[0]
        if pd.isna(orh) or pd.isna(orl):
            continue

        sdf_after = sdf.iloc[1:].copy()
        long_break  = sdf_after[sdf_after["Close"] > orh].head(1)
        short_break = sdf_after[sdf_after["Close"] < orl].head(1)
        if long_break.empty and short_break.empty:
            continue

        if not long_break.empty and not short_break.empty:
            direction = "long" if long_break.index[0] < short_break.index[0] else "short"
            breakout_row = long_break.iloc[0] if direction=="long" else short_break.iloc[0]
        elif not long_break.empty:
            direction = "long"; breakout_row = long_break.iloc[0]
        else:
            direction = "short"; breakout_row = short_break.iloc[0]

        btime = breakout_row.name
        cutoff = btime + timedelta(minutes=max_retest_min)
        after_break = sdf[(sdf.index > btime) & (sdf.index <= cutoff)].copy()
        if after_break.empty:
            continue

        level = orh if direction == "long" else orl
        touch = after_break[(after_break["Low"] <= level) & (after_break["High"] >= level)].head(1)
        if touch.empty:
            continue

        retest_bar = touch.iloc[0]
        retest_time = retest_bar.name

        if retest_confirm_close:
            ok = (retest_bar["Close"] >= level) if direction=="long" else (retest_bar["Close"] <= level)
            if not ok:
                continue

        entry = _apply_slippage(level, slippage_bps, "buy" if direction=="long" else "sell")
        stop = orl if direction=="long" else orh
        rps = abs(entry - stop)
        if rps <= 1e-12:
            continue

        qty = POSITION_SIZE_DOLLARS / entry
        side_mult = 1 if direction=="long" else -1

        t_prices = [(entry + r*rps) if direction=="long" else (entry - r*rps) for r in r_targets]

        sdf_run = sdf[sdf.index >= retest_time].copy()
        exits = []
        for ts, row in sdf_run.iterrows():
            high, low = float(row["High"]), float(row["Low"])
            if low <= stop <= high:
                px = _apply_slippage(stop, slippage_bps, "sell" if direction=="long" else "buy")
                exits.append(("stop", ts, px))
                break
            hit = False
            for i, tp in enumerate(t_prices):
                if low <= tp <= high:
                    px = _apply_slippage(tp, slippage_bps, "sell" if direction=="long" else "buy")
                    exits.append((f"tp{i+1}", ts, px))
                    hit = True
            if hit:
                break

        if not exits:
            last = sdf_run.iloc[-1]
            px = _apply_slippage(float(last["Close"]), slippage_bps, "sell" if direction=="long" else "buy")
            exits.append(("eod", sdf_run.index[-1], px))

        n_parts = len(exits)
        qty_each = qty / n_parts
        cash_pnl = sum((px - entry) * side_mult * qty_each for _, _, px in exits) - FEES_PER_TRADE

        trades.append({
            "Ticker": sdf["Ticker"].iloc[0] if "Ticker" in sdf.columns else "",
            "Session": ses,
            "ORH": float(orh), "ORL": float(orl),
            "Direction": direction,
            "EntryTime": retest_time,
            "Entry": float(entry), "Stop": float(stop),
            "Targets": [float(x) for x in t_prices],
            "Exits": [(str(t[0]), t[1], float(t[2])) for t in exits],
            "Qty": float(qty),
            "PnL_$": float(cash_pnl),
            "R_multiple": float(cash_pnl / (rps * qty))
        })

    trade_log = pd.DataFrame(trades)
    if trade_log.empty:
        eq = pd.DataFrame(columns=["Session","CumPnL_$"])
        return trade_log, eq

    equity = (trade_log.groupby("Session")["PnL_$"].sum()
              .sort_index().cumsum().reset_index().rename(columns={"PnL_$":"CumPnL_$"}))
    return trade_log, equity

# =============================
# Plotting (equity + losing days + drawdown + optional bear shading)
# =============================
def _safe_import_matplotlib():
    try:
        import matplotlib.pyplot as plt
        return plt
    except Exception:
        return None

def _load_spy_daily_for_shading(folder: str) -> pd.DataFrame:
    """Optional: if SPY.parquet exists (any frequency), build daily close and drawdown."""
    spy_path = os.path.join(folder, "SPY.parquet")
    if not os.path.exists(spy_path):
        return pd.DataFrame()
    try:
        spy = load_parquet_15m(spy_path)  # if 15m available; otherwise fallback below
        if spy.empty:
            spy_raw = pd.read_parquet(spy_path, engine="pyarrow")
            spy_raw = _choose_dt_index(spy_raw)
            if spy_raw.index.tz is None:
                spy_raw = spy_raw.tz_localize("UTC").tz_convert(TZ)
            else:
                spy_raw = spy_raw.tz_convert(TZ)
            spy_raw = _standardize_ohlcv_columns(spy_raw)
            spy = spy_raw
        spy = spy.sort_index()
        dclose = spy["Close"].resample("1D").last().dropna()
        if dclose.empty:
            return pd.DataFrame()
        dd = (dclose / dclose.cummax() - 1.0)
        out = pd.DataFrame({"SPY_Close": dclose, "SPY_Drawdown": dd})
        return out
    except Exception:
        return pd.DataFrame()

def _bear_spans(df: pd.DataFrame):
    """Yield (start, end) spans where SPY_Drawdown <= -0.20."""
    if df is None or df.empty or "SPY_Drawdown" not in df.columns: return []
    bear = (df["SPY_Drawdown"] <= -0.20).astype(int)
    spans = []
    in_bear, start = False, None
    for ts, val in bear.items():
        if val and not in_bear:
            in_bear, start = True, ts
        elif not val and in_bear:
            in_bear = False
            spans.append((start, ts))
    if in_bear and start is not None:
        spans.append((start, df.index[-1]))
    return spans

def plot_equity_with_losses(trades: pd.DataFrame, folder: str, save_dir: str, show: bool):
    if trades is None or trades.empty:
        print("[plot] No trades — skipping equity plot.")
        return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping plots.")
        return
    os.makedirs(save_dir, exist_ok=True)

    daily = (trades.groupby("Session")["PnL_$"].sum()
                    .sort_index()
                    .reset_index())
    daily["Session"] = pd.to_datetime(daily["Session"])
    daily["CumPnL_$"] = daily["PnL_$"].cumsum()
    daily["Peak"] = daily["CumPnL_$"].cummax()
    daily["Drawdown_$"] = daily["CumPnL_$"] - daily["Peak"]

    spy_daily = _load_spy_daily_for_shading(folder) if SHADE_BEAR_WITH_SPY else pd.DataFrame()
    spans = _bear_spans(spy_daily)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                   gridspec_kw={"height_ratios":[3,1]})

    ax1.plot(daily["Session"], daily["CumPnL_$"], linewidth=1.5, label="Equity (Cum PnL $)")
    losers = daily[daily["PnL_$"] < 0]
    ax1.scatter(losers["Session"], losers["CumPnL_$"], s=15, color="red", alpha=0.8, label="Losing sessions")

    for s, e in spans:
        ax1.axvspan(s, e, color="gray", alpha=0.15, label=None)

    ax1.set_title("ORB 15m Retest — Equity Curve (cum PnL)")
    ax1.set_ylabel("CumPnL ($)")
    ax1.grid(alpha=0.25)
    ax1.legend(loc="best")

    ax2.plot(daily["Session"], daily["Drawdown_$"], linewidth=1.2)
    ax2.fill_between(daily["Session"], daily["Drawdown_$"], 0, alpha=0.2)
    ax2.set_title("Drawdown ($)")
    ax2.set_ylabel("DD ($)")
    ax2.set_xlabel("Session")
    ax2.grid(alpha=0.25)

    out_path = os.path.join(save_dir, "equity_with_losses_and_dd.png")
    plt.tight_layout()
    plt.savefig(out_path, dpi=140)
    if show:
        plt.show()
    plt.close()
    print(f"[plot] saved {out_path}")

def plot_example_days(df_15: pd.DataFrame, trade_log: pd.DataFrame, ticker: str,
                      max_days: int, save_dir: str, show: bool):
    if max_days <= 0 or trade_log.empty: return
    plt = _safe_import_matplotlib()
    if plt is None:
        print("[plot] matplotlib not available; skipping day charts.")
        return
    os.makedirs(save_dir, exist_ok=True)
    sessions = trade_log["Session"].drop_duplicates().sort_values().tolist()[:max_days]
    for ses in sessions:
        sdf = df_15[df_15.index.date == pd.to_datetime(ses).date()]
        if sdf.empty: continue
        tr = trade_log[trade_log["Session"] == ses].iloc[0]
        plt.figure(figsize=(11,5))
        plt.plot(sdf.index, sdf["Close"], label="Close")
        plt.axhline(tr["ORH"], linestyle="--", label="ORH")
        plt.axhline(tr["ORL"], linestyle="--", label="ORL")
        m = "^" if tr["Direction"]=="long" else "v"
        plt.scatter([tr["EntryTime"]], [tr["Entry"]], marker=m, s=80, label="Entry")
        for tag, tstamp, px in tr["Exits"]:
            plt.scatter([tstamp], [px], marker="x", s=80, label=f"Exit {tag}")
        plt.title(f"{ticker} {pd.to_datetime(ses).date()} ORB Retest (15m)")
        plt.legend(); plt.tight_layout()
        out_path = os.path.join(save_dir, f"{ticker}_{pd.to_datetime(ses).date()}.png")
        plt.savefig(out_path, dpi=130)
        if show: plt.show()
        plt.close()
        print(f"[plot] saved {out_path}")

# =============================
# Runner
# =============================
def run_folder(folder: str):
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"Found {len(files)} parquet files.")
    all_trades, all_equity = [], []

    for i, fpath in enumerate(files, 1):
        # ---- FIXED ticker parsing (no AttributeError) ----
        base = os.path.basename(fpath)
        name, _ext = os.path.splitext(base)
        ticker = name.upper().replace("_", "").replace("-", "").replace(".", "")
        print(f"\n== {ticker} ({i}/{len(files)}) ==\n{fpath}")
        try:
            df15 = load_parquet_15m(fpath, tz=TZ)
            if df15.empty:
                print("Skipped (no 15m data after normalization/resample/session filter).")
                continue
            df15["Ticker"] = ticker
            df15 = compute_opening_range(df15)
        except Exception as e:
            print(f"Skipping {ticker}: {e}")
            continue

        tlog, eq = backtest_orb_retest(
            df15,
            max_retest_min=MAX_RETEST_MIN,
            r_targets=R_MULTIPLES,
            dollars=POSITION_SIZE_DOLLARS,
            slippage_bps=SLIPPAGE_BPS,
            fees=FEES_PER_TRADE,
            retest_confirm_close=RETEST_CONFIRM_CLOSE
        )

        if PLOT_EXAMPLES_PER_TICKER and not tlog.empty:
            plot_example_days(df15, tlog, ticker,
                              max_days=PLOT_EXAMPLES_PER_TICKER,
                              save_dir=PLOT_SAVE_DIR,
                              show=PLOT_SHOW)

        if not tlog.empty:
            out = tlog.copy()
            
            out["Targets"] = out["Targets"].apply(lambda xs: ";".join(f"{p:.6f}" for p in xs))
            out["Exits"] = out["Exits"].apply(lambda xs: ";".join(f"{t}|{ts}|{px:.6f}" for (t, ts, px) in xs))
            all_trades.append(out)
        if not eq.empty:
            eq2 = eq.copy(); eq2["Ticker"] = ticker
            all_equity.append(eq2)

        if AUTOSAVE_EVERY and i % AUTOSAVE_EVERY == 0:
            if all_trades:
                pd.concat(all_trades, ignore_index=True).to_csv(TRADES_CSV_ALL, index=False)
                print(f"[autosave] wrote {TRADES_CSV_ALL}")
            if all_equity:
                pd.concat(all_equity, ignore_index=True).to_csv(EQUITY_CSV_ALL, index=False)
                print(f"[autosave] wrote {EQUITY_CSV_ALL}")

        time.sleep(SLEEP_BETWEEN_TICKERS)

    trades = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()
    equity = pd.concat(all_equity, ignore_index=True) if all_equity else pd.DataFrame()
    return trades, equity

# =============================
# Main
# =============================
def main():
    os.makedirs(PLOT_SAVE_DIR, exist_ok=True)

    trades, equity = run_folder(PARQUET_DIR)

    if not trades.empty:
        trades.sort_values(["Ticker","Session"], inplace=True)
        trades.to_csv(TRADES_CSV_ALL, index=False)
        print(f"Saved trades: {TRADES_CSV_ALL}")

        # Equity/losses/drawdown plot
        plot_equity_with_losses(trades, folder=PARQUET_DIR, save_dir=PLOT_SAVE_DIR, show=PLOT_SHOW)
    else:
        print("No trades generated with current settings.")

    if not equity.empty:
        equity.sort_values(["Ticker","Session"], inplace=True)
        equity.to_csv(EQUITY_CSV_ALL, index=False)
        print(f"Saved equity: {EQUITY_CSV_ALL}")

if __name__ == "__main__":
    main()


Found 20 parquet files.

== AAPL (1/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AAPL.parquet

== AMD (2/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMD.parquet

== AMZN (3/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\AMZN.parquet

== CAT (4/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CAT.parquet

== CHWY (5/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CHWY.parquet

== CVNA (6/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\CVNA.parquet

== GLD (7/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GLD.parquet

== GOOGL (8/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GOOGL.parquet

== GS (9/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\GS.parquet

== JPM (10/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\JPM.parquet

== MSFT (11/20) ==
C:\Users\pcagm\OneDrive\Desktop\downloads\parquet\parquet\MSFT.pa